# NOVA RETAIL — Despacho vs Recepción

## Análisis Forense de Variación en la Cadena de Suministro
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Reconciliación operativa entre CEDIS y tiendas  
**Objetivo:** Comparar los registros de despacho del CEDIS contra las recepciones reportadas por tienda para identificar discrepancias sistemáticas incompatibles con error operativo aleatorio.

---

### Propósito de este notebook
Este notebook evalúa la integridad del flujo de mercancía entre el centro de distribución y las tiendas. Su función es detectar:

- discrepancias entre lo despachado y lo recibido
- patrones concentrados por tienda, categoría o valor
- anomalías horarias en la recepción de mercancía
- señales de pérdida selectiva en productos de alto valor

### Datasets involucrados
- `dispatches.csv`
- `receptions.csv`
- `products_sap.csv`
- `stores.csv`

### Nota del analista
La diferencia entre despacho y recepción no prueba fraude por sí sola.  
Pero cuando la variación se concentra en ciertas tiendas, productos y horarios, deja de parecer ruido operativo y se convierte en una prioridad investigativa.

In [ ]:
%pip install plotly pandas nbformat

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

dispatches = pd.read_csv(DATA_PATH / "dispatches.csv")
receptions = pd.read_csv(DATA_PATH / "receptions.csv")
products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
stores = pd.read_csv(DATA_PATH / "stores.csv")

print("Datasets cargados correctamente.")
print("dispatches:", dispatches.shape)
print("receptions:", receptions.shape)
print("products_sap:", products_sap.shape)
print("stores:", stores.shape)

In [ ]:
dispatches.head()

In [ ]:
receptions.head()

In [ ]:
summary = pd.DataFrame({
    "dataset": ["dispatches", "receptions", "products_sap", "stores"],
    "filas": [
        dispatches.shape[0],
        receptions.shape[0],
        products_sap.shape[0],
        stores.shape[0]
    ],
    "columnas": [
        dispatches.shape[1],
        receptions.shape[1],
        products_sap.shape[1],
        stores.shape[1]
    ]
})

summary

In [ ]:
high_value_categories = ["ELECTRONICA", "TELEFONIA", "COMPUTO", "ELECTRODOMESTICOS"]

products_priority = products_sap[
    products_sap["category_sap"].isin(high_value_categories)
].copy()

products_priority = products_priority[
    products_priority["price_sap"] >= products_priority["price_sap"].quantile(0.95)
].copy()

print("SKUs prioritarios:", len(products_priority))
products_priority[["sku_sap", "description_sap", "category_sap", "price_sap"]].head(10)

In [ ]:
dispatches_priority = dispatches.merge(
    products_priority[["sku_sap", "category_sap", "price_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="inner"
)

receptions_priority = receptions.merge(
    products_priority[["sku_sap", "category_sap", "price_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="inner"
)

print("Despachos prioritarios:", dispatches_priority.shape)
print("Recepciones prioritarias:", receptions_priority.shape)

In [ ]:
receptions_priority["unit_type"].value_counts()

In [ ]:
receptions_priority_box_named = receptions_priority_box.merge(
    products_sap[["sku_sap", "description_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="left"
)

receptions_priority_box_named[[
    "sku",
    "description_sap",
    "category_sap",
    "price_sap",
    "quantity_received",
    "unit_type",
    "notes"
]].head(40)

In [ ]:
receptions_priority_box_named[[
    "description_sap",
    "category_sap",
    "price_sap",
    "quantity_received",
    "unit_type",
    "notes"
]].sort_values("price_sap", ascending=False).head(40)

In [ ]:
merged_priority = dispatches_priority.merge(
    receptions_priority,
    left_on="dispatch_id",
    right_on="dispatch_reference",
    how="inner",
    suffixes=("_dispatch", "_reception")
)

# Regla operativa:
# En el universo prioritario, CAJA se trata como unidad individual
merged_priority["quantity_received_normalized"] = merged_priority["quantity_received"]

merged_priority[[
    "dispatch_id",
    "sku_dispatch",
    "category_sap_dispatch",
    "price_sap_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "quantity_received_normalized",
    "notes"
]].head(20)

In [ ]:
merged_priority["quantity_diff"] = (
    merged_priority["quantity_dispatched"] - merged_priority["quantity_received_normalized"]
)

merged_priority["pct_diff"] = np.where(
    merged_priority["quantity_dispatched"] > 0,
    (merged_priority["quantity_diff"] / merged_priority["quantity_dispatched"]) * 100,
    np.nan
)

merged_priority[[
    "dispatch_id",
    "sku_dispatch",
    "quantity_dispatched",
    "quantity_received_normalized",
    "quantity_diff",
    "pct_diff",
    "notes"
]].sort_values("pct_diff", ascending=False).head(30)

In [ ]:
def normalize_priority_quantity(row):
    if row["unit_type"] == "CAJA":
        if row["quantity_received"] == 1 and row["quantity_dispatched"] in [6, 12, 24]:
            return row["quantity_dispatched"]
        else:
            return row["quantity_received"]
    return row["quantity_received"]

merged_priority["quantity_received_normalized_v2"] = merged_priority.apply(
    normalize_priority_quantity,
    axis=1
)

merged_priority["quantity_diff_v2"] = (
    merged_priority["quantity_dispatched"] - merged_priority["quantity_received_normalized_v2"]
)

merged_priority["pct_diff_v2"] = np.where(
    merged_priority["quantity_dispatched"] > 0,
    (merged_priority["quantity_diff_v2"] / merged_priority["quantity_dispatched"]) * 100,
    np.nan
)

merged_priority[[
    "dispatch_id",
    "sku_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "quantity_received_normalized_v2",
    "quantity_diff_v2",
    "pct_diff_v2",
    "notes"
]].sort_values("pct_diff_v2", ascending=False).head(30)

In [ ]:
def normalize_priority_quantity_v3(row):
    if row["unit_type"] == "CAJA":
        if row["quantity_received"] == 1 and row["quantity_dispatched"] > 1:
            return row["quantity_dispatched"]
        else:
            return row["quantity_received"]
    return row["quantity_received"]

merged_priority["quantity_received_normalized_v3"] = merged_priority.apply(
    normalize_priority_quantity_v3,
    axis=1
)

merged_priority["quantity_diff_v3"] = (
    merged_priority["quantity_dispatched"] - merged_priority["quantity_received_normalized_v3"]
)

merged_priority["pct_diff_v3"] = np.where(
    merged_priority["quantity_dispatched"] > 0,
    (merged_priority["quantity_diff_v3"] / merged_priority["quantity_dispatched"]) * 100,
    np.nan
)

merged_priority[[
    "dispatch_id",
    "sku_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "quantity_received_normalized_v3",
    "quantity_diff_v3",
    "pct_diff_v3",
    "notes"
]].sort_values("pct_diff_v3", ascending=False).head(30)

In [ ]:
merged_priority_named = merged_priority.merge(
    products_sap[["sku_sap", "description_sap"]],
    left_on="sku_dispatch",
    right_on="sku_sap",
    how="left"
)

merged_priority_named[[
    "dispatch_id",
    "description_sap",
    "category_sap_dispatch",
    "price_sap_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "notes"
]].head(50)

In [ ]:
merged_priority_named = merged_priority.merge(
    products_sap[["sku_sap", "description_sap"]],
    left_on="sku_dispatch",
    right_on="sku_sap",
    how="left"
)

In [ ]:
high_diff_raw = merged_priority_named[
    (merged_priority_named["unit_type"] == "CAJA") &
    (merged_priority_named["quantity_received"] == 1) &
    (merged_priority_named["quantity_dispatched"] > 1)
].copy()

In [ ]:
high_diff_raw[[
    "dispatch_id",
    "description_sap",
    "category_sap_dispatch",
    "price_sap_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "notes"
]].head(100)

In [ ]:
high_diff_raw["notes"].value_counts(dropna=False)

In [ ]:
def classify_anomaly_type(row):
    note = str(row["notes"]).strip().lower()

    if "producto sin codigo" in note:
        return "CATALOGO_INVISIBLE"
    elif "faltante parcial" in note:
        return "INCIDENCIA_OPERATIVA_FALTANTE"
    elif "caja dañada" in note:
        return "INCIDENCIA_OPERATIVA_DAÑO"
    elif "revisar con proveedor" in note:
        return "INCIDENCIA_PROVEEDOR"
    elif row["unit_type"] == "CAJA" and row["quantity_received"] == 1 and row["quantity_dispatched"] > 1:
        return "PROBABLE_ERROR_UOM"
    else:
        return "OTRO"

high_diff_raw["anomaly_type"] = high_diff_raw.apply(classify_anomaly_type, axis=1)

high_diff_raw["anomaly_type"].value_counts()

In [ ]:
high_diff_raw[[
    "dispatch_id",
    "description_sap",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "notes",
    "anomaly_type"
]].head(50)

In [ ]:
risk_by_store = (
    high_diff_raw
    .groupby(["store_id_dispatch", "anomaly_type"])
    .size()
    .reset_index(name="casos")
)

risk_by_store.sort_values(["casos"], ascending=False).head(30)

In [ ]:
risk_pivot = risk_by_store.pivot(
    index="store_id_dispatch",
    columns="anomaly_type",
    values="casos"
).fillna(0)

risk_pivot["total_anomalias"] = risk_pivot.sum(axis=1)
risk_pivot = risk_pivot.sort_values("total_anomalias", ascending=False)

risk_pivot.head(20)

In [ ]:
top_risk_stores = risk_pivot.head(15).reset_index()
top_risk_stores["store_id_dispatch"] = top_risk_stores["store_id_dispatch"].astype(str)

plot_df = top_risk_stores.sort_values("total_anomalias", ascending=True).copy()

fig = px.bar(
    plot_df,
    x="total_anomalias",
    y="store_id_dispatch",
    orientation="h",
    text="total_anomalias"
)

fig.update_traces(
    marker=dict(
        color=plot_df["total_anomalias"],
        colorscale=[
            [0.00, "#3B82F6"],
            [0.35, "#06B6D4"],
            [0.60, "#8B5CF6"],
            [1.00, "#EC4899"]
        ],
        line=dict(color="rgba(255,255,255,0.18)", width=1.2)
    ),
    textposition="outside",
    textfont=dict(color="white", size=15, family="Arial Black"),
    hovertemplate="<b>Tienda %{y}</b><br>Total anomalías: %{x}<extra></extra>"
)

fig.update_layout(
    title={
        "text": "Top 15 tiendas prioritarias por volumen de anomalías",
        "x": 0.03,
        "xanchor": "left",
        "font": dict(size=30, color="white", family="Arial Black")
    },
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white", size=14, family="Arial"),
    xaxis=dict(
        title="Total de anomalías",
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False,
        title_font=dict(size=18, color="white"),
        tickfont=dict(size=13, color="white")
    ),
    yaxis=dict(
        title="Store ID",
        showgrid=False,
        categoryorder="total ascending",
        title_font=dict(size=18, color="white"),
        tickfont=dict(size=13, color="white")
    ),
    margin=dict(l=110, r=70, t=90, b=60),
    height=720,
    coloraxis_showscale=False
)

fig.add_annotation(
    text="Ranking de tiendas con mayor concentración de eventos anómalos en productos prioritarios",
    xref="paper", yref="paper",
    x=0.03, y=1.06,
    showarrow=False,
    font=dict(size=14, color="rgba(255,255,255,0.75)", family="Arial")
)

fig.show()

In [ ]:
top_5_share = risk_pivot.head(5)["total_anomalias"].sum() / risk_pivot["total_anomalias"].sum() * 100
top_10_share = risk_pivot.head(10)["total_anomalias"].sum() / risk_pivot["total_anomalias"].sum() * 100

print(f"Participación del Top 5 tiendas: {top_5_share:.2f}%")
print(f"Participación del Top 10 tiendas: {top_10_share:.2f}%")

In [ ]:
cols_investigativas = [
    "CATALOGO_INVISIBLE",
    "INCIDENCIA_OPERATIVA_DAÑO",
    "INCIDENCIA_OPERATIVA_FALTANTE",
    "INCIDENCIA_PROVEEDOR"
]

risk_investigative = risk_pivot[cols_investigativas].copy()
risk_investigative["total_investigativo"] = risk_investigative[cols_investigativas].sum(axis=1)
risk_investigative = risk_investigative.sort_values("total_investigativo", ascending=False)

risk_investigative.head(20)

In [ ]:
top_5_inv = risk_investigative.head(5)["total_investigativo"].sum() / risk_investigative["total_investigativo"].sum() * 100
top_10_inv = risk_investigative.head(10)["total_investigativo"].sum() / risk_investigative["total_investigativo"].sum() * 100

print(f"Participación investigativa del Top 5 tiendas: {top_5_inv:.2f}%")
print(f"Participación investigativa del Top 10 tiendas: {top_10_inv:.2f}%")

In [ ]:
merged_priority_units = merged_priority_named[
    merged_priority_named["unit_type"] == "UNIDAD"
].copy()

print("Registros prioritarios en UNIDAD:", len(merged_priority_units))
merged_priority_units[[
    "dispatch_id",
    "description_sap",
    "category_sap_dispatch",
    "quantity_dispatched",
    "quantity_received",
    "unit_type",
    "notes"
]].head(20)

In [ ]:
severity_by_store = (
    merged_priority_units
    .groupby("store_id_dispatch")[["quantity_dispatched", "quantity_received"]]
    .sum()
    .reset_index()
)

severity_by_store["quantity_diff"] = (
    severity_by_store["quantity_dispatched"] - severity_by_store["quantity_received"]
)

severity_by_store["pct_diff"] = np.where(
    severity_by_store["quantity_dispatched"] > 0,
    (severity_by_store["quantity_diff"] / severity_by_store["quantity_dispatched"]) * 100,
    np.nan
)

severity_by_store = severity_by_store.sort_values("pct_diff", ascending=False)

severity_by_store.head(20)

In [ ]:
plot_df = severity_by_store.head(15).copy()
plot_df["store_id_dispatch"] = plot_df["store_id_dispatch"].astype(str)

fig = px.bar(
    plot_df.sort_values("pct_diff", ascending=True),
    x="pct_diff",
    y="store_id_dispatch",
    orientation="h",
    text="pct_diff"
)

fig.update_traces(
    marker=dict(
        color=plot_df.sort_values("pct_diff", ascending=True)["pct_diff"],
        colorscale=[
            [0.00, "#22C55E"],
            [0.35, "#EAB308"],
            [0.70, "#F97316"],
            [1.00, "#EF4444"]
        ],
        line=dict(color="rgba(255,255,255,0.18)", width=1.2)
    ),
    texttemplate="%{text:.1f}%",
    textposition="outside",
    textfont=dict(color="white", size=14, family="Arial Black"),
    hovertemplate="<b>Tienda %{y}</b><br>Diferencia porcentual: %{x:.2f}%<extra></extra>"
)

fig.update_layout(
    title={
        "text": "Top 15 tiendas por severidad de discrepancia (solo UNIDAD)",
        "x": 0.03,
        "xanchor": "left",
        "font": dict(size=28, color="white", family="Arial Black")
    },
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white", size=14, family="Arial"),
    xaxis=dict(
        title="% de diferencia entre despacho y recepción",
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),
    yaxis=dict(
        title="Store ID",
        showgrid=False,
        categoryorder="total ascending"
    ),
    margin=dict(l=110, r=70, t=90, b=60),
    height=720,
    coloraxis_showscale=False
)

fig.show()

In [ ]:
severity_by_store_category = (
    merged_priority_units
    .groupby(["store_id_dispatch", "category_sap_dispatch"])[["quantity_dispatched", "quantity_received"]]
    .sum()
    .reset_index()
)

severity_by_store_category["quantity_diff"] = (
    severity_by_store_category["quantity_dispatched"] - severity_by_store_category["quantity_received"]
)

severity_by_store_category["pct_diff"] = np.where(
    severity_by_store_category["quantity_dispatched"] > 0,
    (severity_by_store_category["quantity_diff"] / severity_by_store_category["quantity_dispatched"]) * 100,
    np.nan
)

severity_by_store_category.sort_values("pct_diff", ascending=False).head(30)

In [ ]:
top_severe_stores = severity_by_store.head(10)["store_id_dispatch"].tolist()

severity_focus = severity_by_store_category[
    severity_by_store_category["store_id_dispatch"].isin(top_severe_stores)
].copy()

severity_focus["store_id_dispatch"] = severity_focus["store_id_dispatch"].astype(str)

fig = px.bar(
    severity_focus,
    x="store_id_dispatch",
    y="pct_diff",
    color="category_sap_dispatch",
    barmode="group",
    title="Severidad de discrepancia por tienda y categoría (Top 10 tiendas)",
    labels={
        "store_id_dispatch": "Store ID",
        "pct_diff": "% de diferencia",
        "category_sap_dispatch": "Categoría"
    }
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white"),
    height=650
)

fig.show()

In [ ]:
merged_priority_units["reception_hour"] = (
    pd.to_datetime(merged_priority_units["reception_time"], format="%H:%M", errors="coerce")
    .dt.hour
)

merged_priority_units[[
    "store_id_dispatch",
    "description_sap",
    "reception_time",
    "reception_hour"
]].head(20)

In [ ]:
top_severe_store_ids = severity_by_store.head(10)["store_id_dispatch"].tolist()

hourly_top = merged_priority_units[
    merged_priority_units["store_id_dispatch"].isin(top_severe_store_ids)
]["reception_hour"].value_counts().sort_index()

hourly_rest = merged_priority_units[
    ~merged_priority_units["store_id_dispatch"].isin(top_severe_store_ids)
]["reception_hour"].value_counts().sort_index()

hourly_compare = pd.DataFrame({
    "Top tiendas severas": hourly_top,
    "Resto de tiendas": hourly_rest
}).fillna(0)

hourly_compare

In [ ]:
hourly_compare.reset_index().columns

In [ ]:
hourly_plot = hourly_compare.reset_index()

hourly_plot = hourly_plot.melt(
    id_vars="reception_hour",
    var_name="grupo",
    value_name="conteo"
)

fig = px.bar(
    hourly_plot,
    x="reception_hour",
    y="conteo",
    color="grupo",
    barmode="group",
    title="Distribución horaria de recepciones: tiendas severas vs resto",
    labels={
        "reception_hour": "Hora de recepción",
        "conteo": "Número de recepciones",
        "grupo": "Grupo"
    }
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white"),
    height=650
)

fig.show()

In [ ]:
top_severe_store_ids = severity_by_store.head(10)["store_id_dispatch"].tolist()

top_severe_store_meta = stores[
    stores["store_id"].isin(top_severe_store_ids)
][[
    "store_id",
    "store_name",
    "city",
    "state",
    "system",
    "cedis_route",
    "regional_manager_id",
    "ghost_store"
]].sort_values("store_id")

top_severe_store_meta

In [ ]:
top_severe_store_meta["cedis_route"].value_counts()

In [ ]:
top_severe_store_meta["regional_manager_id"].value_counts()

In [ ]:
early_hour_counts = (
    merged_priority_units[merged_priority_units["reception_hour"] == 5]
    .groupby("store_id_dispatch")
    .size()
    .reset_index(name="recepciones_5am")
    .sort_values("recepciones_5am", ascending=False)
)

early_hour_counts.head(20)

In [ ]:
top_severe_5am = severity_by_store.merge(
    early_hour_counts,
    left_on="store_id_dispatch",
    right_on="store_id_dispatch",
    how="left"
).fillna({"recepciones_5am": 0})

top_severe_5am = top_severe_5am.sort_values("pct_diff", ascending=False)

top_severe_5am.head(15)

In [ ]:
plot_final = top_severe_5am.head(15).copy()
plot_final["store_id_dispatch"] = plot_final["store_id_dispatch"].astype(str)

fig = px.scatter(
    plot_final,
    x="recepciones_5am",
    y="pct_diff",
    size="quantity_diff",
    color="pct_diff",
    text="store_id_dispatch",
    color_continuous_scale=[
        [0.00, "#22C55E"],
        [0.35, "#EAB308"],
        [0.70, "#F97316"],
        [1.00, "#EF4444"]
    ],
    title="Relación entre severidad de discrepancia y recepciones a las 5 AM"
)

fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(color="rgba(255,255,255,0.25)", width=1.2)),
    hovertemplate=(
        "<b>Tienda %{text}</b><br>" +
        "Recepciones 5 AM: %{x}<br>" +
        "Diferencia %: %{y:.2f}<br>" +
        "Diferencia absoluta: %{marker.size}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#1E1E1E",
    paper_bgcolor="#1E1E1E",
    font=dict(color="white", size=14),
    height=700,
    xaxis_title="Número de recepciones a las 5 AM",
    yaxis_title="% de diferencia entre despacho y recepción",
    coloraxis_showscale=False
)

fig.show()

## Cierre ejecutivo — Versión CEO

**Conclusión ejecutiva:**  
El análisis confirma que las discrepancias más severas no están distribuidas al azar: se concentran en un grupo específico de tiendas dentro de la Ruta Norte y coinciden con una ventana de recepción atípica a las 5 AM.

**Implicación estratégica:**  
Ese patrón no es consistente con fricción operativa normal y sugiere una vulnerabilidad localizada en la cadena de custodia.

**Recomendación inmediata:**  
Avanzar de inmediato a una investigación focalizada sobre ese corredor logístico antes de que el patrón se adapte, se opaque o desaparezca.

## Cierre ejecutivo — Versión CFO

**Conclusión ejecutiva:**  
La pérdida no parece estar dispersa en toda la red, sino concentrada en un subconjunto operativo con discrepancias de recepción de entre 13% y 20.5% sobre productos prioritarios.

**Implicación financiera:**  
Esto permite abandonar una limpieza horizontal costosa y migrar hacia una intervención focalizada sobre las tiendas y flujos que realmente están deteriorando valor.

**Recomendación inmediata:**  
Concentrar recursos analíticos y operativos en la Ruta Norte, donde la severidad observada supera ampliamente el rango de merma operativa esperada y ofrece la mejor relación entre esfuerzo e impacto económico.

## Cierre ejecutivo — Versión Legal

**Conclusión ejecutiva:**  
Los datos identifican una anomalía operativa localizada caracterizada por discrepancias materiales entre despacho y recepción, asociadas a una franja horaria atípica y a un corredor logístico específico.

**Límite probatorio actual:**  
Estos hallazgos no constituyen por sí mismos prueba de conducta individual indebida, pero sí establecen una base objetiva para una revisión focalizada de controles, registros y cadena de custodia.

**Recomendación inmediata:**  
Cualquier escalamiento a personas, responsabilidades o medidas disciplinarias deberá apoyarse en una segunda fase de evidencia nominal y en un protocolo formal de preservación de evidencia.

In [ ]:
resumen_dispatch_reception = pd.DataFrame({
    "indicador": [
        "Despachos totales",
        "Recepciones totales",
        "Registros prioritarios analizados",
        "Tiendas severas identificadas",
        "Máxima discrepancia porcentual",
        "Mínima discrepancia en tiendas normales",
        "Tiendas severas con recepciones 5 AM",
        "Ruta dominante en tiendas severas"
    ],
    "valor": [
        len(dispatches),
        len(receptions),
        len(merged_priority_units),
        10,
        f"{severity_by_store['pct_diff'].max():.2f}%",
        f"{severity_by_store['pct_diff'].tail(5).min():.2f}%",
        int((top_severe_5am.head(10)["recepciones_5am"] > 0).sum()),
        "NORTE"
    ]
})

resumen_dispatch_reception

## Conclusiones de la fase despacho vs recepción

### Hallazgos principales
- El cruce entre despachos y recepciones mostró integridad estructural a nivel de llave (`dispatch_id` ↔ `dispatch_reference`).
- La métrica más útil para detectar el patrón relevante no fue la frecuencia de incidencias, sino la **severidad porcentual de discrepancia**.
- En productos prioritarios registrados como `UNIDAD`, emergió un subconjunto de tiendas con diferencias de recepción entre **13% y 20.5%**, muy por encima del rango operativo normal.
- Las tiendas con mayor severidad se concentran completamente en la **Ruta Norte**.
- Ese mismo subconjunto presenta recepciones en la franja de las **05:00**, ausente en las tiendas de comportamiento normal.
- La combinación de **severidad + ruta + horario** constituye una señal operativa localizada e incompatible con ruido distribuido.

### Límite actual
Este notebook no atribuye responsabilidad individual ni prueba fraude por sí mismo. Su función es identificar un patrón operativo priorizable y defendible para escalamiento investigativo.

### Decisión metodológica
La siguiente fase debe concentrarse en profundizar el análisis sobre la Ruta Norte, integrando:
- trazabilidad de usuarios
- metadata de autorización
- comparación por tienda y corredor logístico
- y evidencia nominal bajo control de cadena de custodia

## Próximo paso recomendado

Con base en los hallazgos de esta fase, la siguiente etapa del análisis debe enfocarse en:

1. **Cruzar severidad de discrepancia con usuarios autorizadores y metadatos de despacho**
2. **Analizar la relación entre ruta, horario de recepción y pérdida material**
3. **Separar de forma controlada ruido operativo, error de unidad de medida y señales de vulnerabilidad localizada**
4. **Preparar una matriz de priorización investigativa para escalamiento ejecutivo**

El análisis deja claro que la siguiente pregunta ya no es si existe una anomalía, sino **quiénes y qué controles están posicionados alrededor de ella**.